###### Imports and Settings

In [1]:
import pandas as pd
import numpy as np
import requests
#import io
import pickle
from collections import deque
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

# Comprehensive Plan Data Pull  

The following API calls are designed to streamline the data pulls for the comprehensive plans for any geography. For the sake of simplicity, there are three separate documents for ACS variables, Decennial 2000 SF3 variables, and Decennial SF1 variables. This document contains Decennial SF1 variables.

In [134]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [135]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']

In [136]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

In [137]:
YEARS = ['2000','2010']

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  
+ dg14: ID's 599-644  
+ dg15: ID's 645-690  
+ dg16: ID's 691-736  
+ dg17: ID's 737-782  
+ dg18: ID's 783-828  
**This data guide stops at ID 783 as of 5/18/2022.**

### SF1

In [138]:
dataguide = pd.read_csv('../data guides/DATA GUIDE SF12000.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [139]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]

In [140]:
# ONE 2000
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['SF1 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
one = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                  |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                  |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                  |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                  |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                  |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
one = one.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
one = one.append(state, ignore_index = True)
onepull = one
print('Okay Finished')

Okay Finished


In [165]:
one = onepull

In [143]:
# TWO 2000
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg2['SF1 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg2['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
two = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                  |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                  |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                  |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                  |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                  |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
two = two.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
two = two.append(state, ignore_index = True)
twopull = two
print('Okay Finished')

Okay Finished


In [166]:
two = twopull

In [167]:
last = two

## Joining

In [168]:
one = one.drop(columns = ['StateFIPS','GeoFIPS'])
last = last.drop(columns = 'NAME')

In [169]:
data = one.merge(last, on = 'GEO_ID')

In [170]:
transp = data.transpose()
transp.columns = transp.iloc[0]
transp.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Burns town, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Dickson city, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee",Tennessee
NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Burns town, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Dickson city, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee",Tennessee
GEO_ID,0500000US47161,0500000US47125,0500000US47083,0500000US47085,0500000US47043,0500000US47021,0500000US47147,0500000US47165,0500000US47037,0500000US47189,0500000US47169,0500000US47187,0500000US47149,0500000US47119,1600000US4702180,1600000US4709880,1600000US4713080,1600000US4715160,1600000US4720620,1600000US4722360,1600000US4728540,1600000US4739660,1600000US4741200,1600000US4741520,1600000US4744840,1600000US4750780,1600000US4751560,1600000US4752820,1600000US4757480,1600000US4759560,1600000US4769080,1600000US4769420,1600000US4776860,1600000US4778320,1600000US4778560,1600000US4779980,0400000US47
pop,12370,134768,8088,17929,43156,35912,54433,130449,569891,88809,7259,126638,182023,69498,3641,1366,1153,103455,12244,464,23230,2773,18687,20235,1702,12366,68816,1905,2146,2934,185,25569,310,1358,4028,2142,5689283
agebysex_total_series,12370,134768,8088,17929,43156,35912,54433,130449,569891,88809,7259,126638,182023,69498,3641,1366,1153,103455,12244,464,23230,2773,18687,20235,1702,12366,68816,1905,2146,2934,185,25569,310,1358,4028,2142,5689283
age_total_male,6158,67775,3999,8819,21158,17981,27051,63876,275865,43830,3573,62358,90585,33782,1782,677,609,51950,5685,223,11051,1378,9215,9621,796,6037,34184,924,1089,1476,97,12516,151,642,1905,1076,2770275


In [171]:
transp = transp.loc[transp['Stewart County, Tennessee'] != 'Stewart County, Tennessee']
transp = transp.loc[transp['Stewart County, Tennessee'] != '0500000US47161']

In [172]:
numcols = list(transp.columns)
numcols
transp[numcols] = transp[numcols].astype(float)

In [173]:
data = transp

In [174]:
GNRCCounties = [data['Stewart County, Tennessee'],data['Montgomery County, Tennessee'],data['Houston County, Tennessee'],data['Humphreys County, Tennessee'],
                data['Dickson County, Tennessee'],data['Cheatham County, Tennessee'],data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],
                data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],data['Trousdale County, Tennessee'],data['Williamson County, Tennessee'],
                data['Rutherford County, Tennessee']]
data['GNRC'] = sum(GNRCCounties)

In [175]:
MPOCounties = [data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],
               data['Williamson County, Tennessee'],data['Rutherford County, Tennessee'],data['Maury County, Tennessee']]
data['MPO'] = sum(MPOCounties)

In [176]:
RuthInc = [data['Eagleville city, Tennessee'],data['La Vergne city, Tennessee'],data['Murfreesboro city, Tennessee'],data['Smyrna town, Tennessee']]
data['Rutherford Incorporated'] = sum(RuthInc)
data['Rutherford Unincorporated'] = data['Rutherford County, Tennessee'] - data['Rutherford Incorporated']
WilsonInc = [data['Lebanon city, Tennessee'],data['Mount Juliet city, Tennessee'],data['Watertown city, Tennessee']]
data['Wilson Incorporated'] = sum(WilsonInc)
data['Wilson Unincorporated'] = data['Wilson County, Tennessee'] - data['Wilson Incorporated']
CheathInc = [data['Ashland City town, Tennessee'],data['Kingston Springs town, Tennessee'],data['Pegram town, Tennessee'],data['Pleasant View city, Tennessee']]
data['Cheatham Incorporated'] = sum(CheathInc)
data['Cheatham Unincorporated'] = data['Cheatham County, Tennessee'] - data['Cheatham Incorporated']
DicksInc = [data['Burns town, Tennessee'],data['Charlotte town, Tennessee'],data['Dickson city, Tennessee'],data['Slayden town, Tennessee'],
            data['Vanleer town, Tennessee'],data['White Bluff town, Tennessee']]
data['Dickson Incorporated'] = sum(DicksInc)
data['Dickson Unincorporated'] = data['Dickson County, Tennessee'] - data['Dickson Incorporated']
HumphInc = [data['McEwen city, Tennessee'],data['New Johnsonville city, Tennessee'],data['Waverly city, Tennessee']]
data['Humphreys Incorporated'] = sum(HumphInc)
data['Humphreys Unincorporated'] = data['Humphreys County, Tennessee'] - data['Humphreys Incorporated']
data['Montgomery Incorporated'] = data['Clarksville city, Tennessee']
data['Montgomery Unincorporated'] = data['Montgomery County, Tennessee'] - data['Montgomery Incorporated']
# data['Trousdale Incorporated'] = data['Hartsville/Trousdale County, Tennessee']
# data['Trousdale Unincorporated'] = data['Trousdale County, Tennessee'] - data['Trousdale Incorporated']

In [177]:
data = data.transpose().reset_index()

In [178]:
data.head()

,NAME,pop,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_total_series,raceeth_total_oneracealone,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,units_allhousing,occupancy_total_series,occupancy_occupiedunits,occupancy_vacantunits,tenure_total_series,tenure_owneroccunits,tenure_renteroccunits,hhtype_total_series,hhtype_oneperson,hhtype_oneperson_male,hhtype_oneperson_female,hhtype_twoormore,hhtype_twoormore_family,hhtype_twoormore_family_marriedcouplefam,hhtype_twoormore_family_marriedcouplefam_ownchildrenunder18,hhtype_twoormore_family_marriedcouplefam_noownchildrenunder18,hhtype_twoormore_family_otherfam,hhtype_twoormore_family_otherfam_malenospouse,hhtype_twoormore_family_otherfam_malenospouse_ownchildrenunder18,hhtype_twoormore_family_otherfam_malenospouse_noownchildrenunder18,hhtype_twoormore_family_otherfam_femalenospouse,hhtype_twoormore_family_otherfam_femalenospouse_ownchildrenunder18,hhtype_twoormore_family_otherfam_femalenospouse_noownchildrenunder18,hhtype_twoormore_nonfamily,hhtype_twoormore_nonfamily_male,hhtype_twoormore_nonfamily_female,StateFIPS,GeoFIPS
0,"Stewart County, Tennessee",12370.0,12370.0,6158.0,374.0,430.0,465.0,267.0,168.0,67.0,70.0,167.0,359.0,418.0,520.0,454.0,456.0,450.0,366.0,125.0,173.0,111.0,191.0,203.0,173.0,83.0,68.0,6212.0,353.0,420.0,417.0,234.0,161.0,42.0,61.0,187.0,378.0,391.0,488.0,499.0,445.0,397.0,371.0,144.0,210.0,121.0,165.0,257.0,193.0,136.0,142.0,12370.0,12234.0,11785.0,159.0,75.0,180.0,6.0,29.0,136.0,11701.0,124.0,2.49,5977.0,5977.0,4930.0,1047.0,4930.0,3904.0,1026.0,4930.0,1140.0,527.0,613.0,3790.0,3652.0,3071.0,1208.0,1863.0,581.0,183.0,104.0,79.0,398.0,241.0,157.0,138.0,90.0,48.0,47.0,161.0
1,"Montgomery County, Tennessee",134768.0,134768.0,67775.0,5901.0,5751.0,5340.0,2863.0,2128.0,1317.0,1380.0,4050.0,6608.0,6027.0,5842.0,5081.0,3888.0,3200.0,2251.0,772.0,1070.0,654.0,1011.0,1224.0,704.0,417.0,296.0,66993.0,5552.0,5335.0,4878.0,2709.0,1987.0,1148.0,1168.0,3374.0,5987.0,5619.0,5840.0,5196.0,4018.0,3294.0,2529.0,881.0,1285.0,779.0,1068.0,1440.0,1279.0,844.0,783.0,134768.0,130849.0,98611.0,25848.0,709.0,2455.0,287.0,2939.0,3919.0,95515.0,6960.0,2.70,52167.0,52167.0,48330.0,3837.0,48330.0,30700.0,17630.0,48330.0,9740.0,4727.0,5013.0,38590.0,35964.0,28371.0,14702.0,13669.0,7593.0,1701.0,1011.0,690.0,5892.0,3951.0,1941.0,2626.0,1736.0,890.0,47.0,125.0
2,"Houston County, Tennessee",8088.0,8088.0,3999.0,268.0,311.0,266.0,174.0,86.0,36.0,39.0,146.0,264.0,256.0,255.0,300.0,280.0,266.0,234.0,83.0,150.0,73.0,112.0,158.0,112.0,77.0,53.0,4089.0,273.0,243.0,298.0,137.0,78.0,37.0,43.0,127.0,244.0,240.0,277.0,271.0,293.0,274.0,260.0,92.0,137.0,92.0,112.0,180.0,151.0,113.0,117.0,8088.0,8011.0,7650.0,268.0,15.0,10.0,5.0,63.0,77.0,7611.0,101.0,2.46,3901.0,3901.0,3216.0,685.0,3216.0,2476.0,740.0,3216.0,815.0,368.0,447.0,2401.0,2300.0,1833.0,746.0,1087.0,467.0,134.0,59.0,75.0,333.0,194.0,139.0,101.0,68.0,33.0,47.0,83.0
3,"Humphreys County, Tennessee",17929.0,17929.0,8819.0,567.0,610.0,660.0,402.0,214.0,108.0,110.0,280.0,537.0,536.0,651.0,701.0,635.0,624.0,589.0,202.0,275.0,153.0,240.0,312.0,225.0,114.0,74.0,9110.0,505.0,597.0,584.0,357.0,198.0,89.0,89.0,278.0,602.0,571.0,669.0,668.0,

# Come back for hartsville/trousdale.... not there for some reason

# Calculations

## Population + Projections Summary: 

In [179]:
data['Population'] = data['agebysex_total_series']
data = data.drop(columns = ['agebysex_total_series','pop'])

## Age Sex Race Summary:

### Age Brackets
+ M and F U5, 5 to 9, 10 to 14, 15 to 17, 18 to 24, 25 to 34, 35 to 44, 45 to 54, 55 to 64, 65 to 74, 75 to 84, 85+  
+ age brackets: under 18, 18 to 54, 55+  
+ 65+ 

In [180]:
#small groups m and f
data['Male Under 5'] = data['age_m_u5']
data['Female Under 5'] = data['age_f_u5']
data['Male 5 to 9'] = data['age_m_5to9']
data['Female 5 to 9'] = data['age_f_5to9']
data['Male 10 to 14'] = data['age_m_10to14']
data['Female 10 to 14'] = data['age_f_10to14']
data['Male 15 to 17'] = data['age_m_15to17']
data['Female 15 to 17'] = data['age_f_15to17']
data['Male 18 to 24'] = data['age_m_18to19']+data['age_m_20']+data['age_m_21']+data['age_m_22to24']
data['Female 18 to 24'] = data['age_f_18to19']+data['age_f_20']+data['age_f_21']+data['age_f_22to24']
data['Male 25 to 34'] = data['age_m_25to29']+data['age_m_30to34']
data['Female 25 to 34'] = data['age_f_25to29']+data['age_f_30to34']
data['Male 35 to 44'] = data['age_m_35to39']+data['age_m_40to44']
data['Female 35 to 44'] = data['age_f_35to39']+data['age_f_40to44']
data['Male 45 to 54'] = data['age_m_45to49']+data['age_m_50to54']
data['Female 45 to 54'] = data['age_f_45to49']+data['age_f_50to54']
data['Male 55 to 64'] = data['age_m_55to59']+data['age_m_60to61']+data['age_m_62to64']
data['Female 55 to 64'] = data['age_f_55to59']+data['age_f_60to61']+data['age_f_62to64']
data['Male 65 to 74'] = data['age_m_65to66']+data['age_m_67to69']+data['age_m_70to74']
data['Female 65 to 74'] = data['age_f_65to66']+data['age_f_67to69']+data['age_f_70to74']
data['Male 75 to 84'] = data['age_m_75to79']+data['age_m_80to84']
data['Female 75 to 84'] = data['age_f_75to79']+data['age_f_80to84']
data['Male 85 and Older'] = data['age_m_85+']
data['Female 85 and Older'] = data['age_f_85+']

#age brackets
u18list = [data['Male Under 5'],data['Female Under 5'],data['Male 5 to 9'],data['Female 5 to 9'],data['Male 10 to 14'],data['Female 10 to 14'],data['Male 15 to 17'],
           data['Female 15 to 17']]
data['Age:Under 18'] = sum(u18list)
eighteento54list = [data['Male 18 to 24'],data['Female 18 to 24'],data['Male 25 to 34'],data['Female 25 to 34'],data['Male 35 to 44'],data['Female 35 to 44'],
              data['Male 45 to 54'],data['Female 45 to 54']]
data['Age:18 to 24'] = sum(eighteento54list)
fifty5pluslist = [data['Male 55 to 64'],data['Female 55 to 64'],data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],
                  data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:55 and Older'] = sum(fifty5pluslist)

#65+
sixty5pluslist = [data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:65 and Older'] = sum(sixty5pluslist)

cols = ['age_total_male','age_total_female','age_m_u5','age_m_5to9','age_m_10to14','age_m_15to17','age_m_18to19','age_m_20','age_m_21','age_m_22to24','age_m_25to29','age_m_30to34','age_m_35to39',
        'age_m_40to44','age_m_45to49','age_m_50to54','age_m_55to59','age_m_60to61','age_m_62to64','age_m_65to66','age_m_67to69','age_m_70to74','age_m_75to79',
        'age_m_80to84','age_m_85+','age_f_u5','age_f_5to9','age_f_10to14','age_f_15to17','age_f_18to19','age_f_20','age_f_21','age_f_22to24','age_f_25to29',
        'age_f_30to34','age_f_35to39','age_f_40to44','age_f_45to49','age_f_50to54','age_f_55to59','age_f_60to61','age_f_62to64','age_f_65to66','age_f_67to69',
        'age_f_70to74','age_f_75to79','age_f_80to84','age_f_85+']
data = data.drop(columns = cols)

### Race/Ethnicity #s   
+ White Alone  
+ Black/African American Alone  
+ American Indian Alaska Native Alone  
+ Asian Alone  
+ Native Hawaiian/Other Pacific Islander Alone  
+ Some Other Race Alone  
+ Two or More Races  
+ Hispanic/Latino  
+ White, Not Hispanic/Latino  
+ Total Minority (non-white, non Hispanic/Latino)

In [181]:
#White Alone
data['White Alone'] = data['raceeth_white_alone']
data['White Alone %'] = data['White Alone']/data['Population']
#Black or African American Alone 
data['Black or African American Alone'] = data['raceeth_blackafricanamerican_alone']
data['Black or African American Alone %'] = data['Black or African American Alone']/data['Population']
#American Indian and Alaska Native Alone
data['American Indian Alaska Native Alone'] = data['raceeth_americanindianalaskanative_alone']
data['American Indian Alaska Native Alone %'] = data['American Indian Alaska Native Alone']/data['Population']
#Asian Alone
data['Asian Alone'] = data['raceeth_asian_alone']
data['Asian Alone %'] = data['Asian Alone']/data['Population']
#Native Hawaiian Other Pacific Islander Alone
data['Native Hawaiian Other Pacific Islander Alone'] = data['raceeth_nativehawaiianotherpacificislander_alone']
data['Native Hawaiian Other Pacific Islander Alone %'] = data['Native Hawaiian Other Pacific Islander Alone']/data['Population']
#Some Other Race Alone
data['Some Other Race Alone'] = data['raceeth_someotherrace_alone']
data['Some Other Race Alone %'] = data['Some Other Race Alone']/data['Population']
#Two or More Races
data['Two or More Races'] = data['raceeth_twoormoreraces_alone']
data['Two or More Races %'] = data['Two or More Races']/data['Population']
#Hispanic or Latino
data['Hispanic or Latino'] = data['raceeth_hispanicorlatino']
data['Hispanic or Latino %'] = data['Hispanic or Latino']/data['Population']
#Data Minority
data['Minority'] = data['Population'] - data['raceeth_whitealone_nothispanicorlatino']
data['Minority %'] = data['Minority']/data['Population']
cols = ['raceeth_total_series','raceeth_white_alone','raceeth_blackafricanamerican_alone','raceeth_americanindianalaskanative_alone','raceeth_asian_alone',
        'raceeth_nativehawaiianotherpacificislander_alone','raceeth_someotherrace_alone','raceeth_twoormoreraces_alone','raceeth_hispanicorlatino',
        'raceeth_whitealone_nothispanicorlatino','raceeth_total_oneracealone']
data = data.drop(columns = cols)

## Household Summary:  

### Households by Household Type  
+ Total Households  
+ Family Households  
+ Family Households: Married-Couple Family  
+ Family Households: Other Family  
+ Family Households: Other Family: Male Householder no Wife Present  
+ Family Households: Other Family: Female Householder no Husband Present  
+ Nonfamily Households  
+ Nonfamily Household: Male Householder  
+ Nonfamily Household: Female Householder  

*They don't have the nonfamily male or female - replaced by Householder alone or not alone*

In [182]:
data['Total Households'] = data['hhtype_total_series']
data['Family Households'] = data['hhtype_twoormore_family']
data['Family Households: Married Couple Family'] = data['hhtype_twoormore_family_marriedcouplefam']
data['Family Households: Not Married Couple Family'] = data['hhtype_twoormore_family_otherfam']
data['Family Households: Not Married Couple: Male no Spouse'] = data['hhtype_twoormore_family_otherfam_malenospouse']
data['Family Households: Not Married Couple: Female no Spouse'] = data['hhtype_twoormore_family_otherfam_femalenospouse']
data['Nonfamily Households'] = data['hhtype_twoormore_nonfamily'] + data['hhtype_oneperson']
data['Nonfamily Households: Householder Alone'] = data['hhtype_oneperson']
data['Nonfamily Households: Householder not Alone'] = data['hhtype_twoormore_nonfamily']

cols = ['hhtype_total_series','hhtype_oneperson','hhtype_oneperson_male','hhtype_oneperson_female','hhtype_twoormore','hhtype_twoormore_family',
        'hhtype_twoormore_family_marriedcouplefam','hhtype_twoormore_family_marriedcouplefam_ownchildrenunder18',
        'hhtype_twoormore_family_marriedcouplefam_noownchildrenunder18','hhtype_twoormore_family_otherfam','hhtype_twoormore_family_otherfam_malenospouse',
        'hhtype_twoormore_family_otherfam_malenospouse_ownchildrenunder18','hhtype_twoormore_family_otherfam_malenospouse_noownchildrenunder18',
        'hhtype_twoormore_family_otherfam_femalenospouse','hhtype_twoormore_family_otherfam_femalenospouse_ownchildrenunder18',
        'hhtype_twoormore_family_otherfam_femalenospouse_noownchildrenunder18','hhtype_twoormore_nonfamily','hhtype_twoormore_nonfamily_male',
        'hhtype_twoormore_nonfamily_female']
data = data.drop(columns = cols)

### Average Household Size 
Median - drop for incorporated and unincorporated

In [183]:
data['Average Household Size'] = data['hhsize_avg']
data = data.drop(columns = ['hhsize_avg'])

### Occupancy Status, and Percentages  
+ Occupancy Total Households
+ Occupied  
+ Vacant  

In [184]:
data['Occupancy:Total Households'] = data['occupancy_total_series']
data['Occupancy:Occupied Units'] = data['occupancy_occupiedunits']
data['Occupancy%:Occupied Units'] = percent(data['Occupancy:Occupied Units'], data['Occupancy:Total Households'])
data['Occupancy:Vacant Units'] = data['occupancy_vacantunits']
data['Occupancy%:Vacant Units'] = percent(data['Occupancy:Vacant Units'], data['Occupancy:Total Households'])

cols = ['occupancy_total_series','occupancy_occupiedunits','occupancy_vacantunits']
data = data.drop(columns = cols)

### Tenure, and Percentages  
+ Tenure Total Households  
+ Owners  
+ Renters

In [185]:
data['Tenure:Total Households'] = data['tenure_total_series']
data['Tenure:Owners'] = data['tenure_owneroccunits']
data['Tenure%:Owners'] = percent(data['Tenure:Owners'], data['Tenure:Total Households'])
data['Tenure:Renters'] = data['tenure_renteroccunits']
data['Tenure%:Renters'] = percent(data['Tenure:Renters'], data['Tenure:Total Households'])
cols = ['units_allhousing','tenure_total_series','tenure_owneroccunits','tenure_renteroccunits']
data = data.drop(columns = cols)

In [186]:
data.head()

,NAME,StateFIPS,GeoFIPS,Population,Male Under 5,Female Under 5,Male 5 to 9,Female 5 to 9,Male 10 to 14,Female 10 to 14,Male 15 to 17,Female 15 to 17,Male 18 to 24,Female 18 to 24,Male 25 to 34,Female 25 to 34,Male 35 to 44,Female 35 to 44,Male 45 to 54,Female 45 to 54,Male 55 to 64,Female 55 to 64,Male 65 to 74,Female 65 to 74,Male 75 to 84,Female 75 to 84,Male 85 and Older,Female 85 and Older,Age:Under 18,Age:18 to 24,Age:55 and Older,Age:65 and Older,White Alone,White Alone %,Black or African American Alone,Black or African American Alone %,American Indian Alaska Native Alone,American Indian Alaska Native Alone %,Asian Alone,Asian Alone %,Native Hawaiian Other Pacific Islander Alone,Native Hawaiian Other Pacific Islander Alone %,Some Other Race Alone,Some Other Race Alone %,Two or More Races,Two or More Races %,Hispanic or Latino,Hispanic or Latino %,Minority,Minority %,Total Households,Family Households,Family Households: Married Couple Family,Family Households: Not Married Couple Family,Family Households: Not Married Couple: Male no Spouse,Family Households: Not Married Couple: Female no Spouse,Nonfamily Households,Nonfamily Households: Householder Alone,Nonfamily Households: Householder not Alone,Average Household Size,Occupancy:Total Households,Occupancy:Occupied Units,Occupancy%:Occupied Units,Occupancy:Vacant Units,Occupancy%:Vacant Units,Tenure:Total Households,Tenure:Owners,Tenure%:Owners,Tenure:Renters,Tenure%:Renters
0,"Stewart County, Tennessee",47.0,161.0,12370.0,374.0,353.0,430.0,420.0,465.0,417.0,267.0,234.0,472.0,451.0,777.0,769.0,974.0,987.0,906.0,842.0,664.0,725.0,505.0,543.0,256.0,329.0,68.0,142.0,2960.0,6178.0,3232.0,1843.0,11785.0,0.952708,159.0,0.012854,75.0,0.006063,180.0,0.014551,6.0,0.000485,29.0,0.002344,136.0,0.010994,124.0,0.010024,669.0,0.054082,4930.0,3652.0,3071.0,581.0,183.0,398.0,1278.0,1140.0,138.0,2.49,5977.0,4930.0,82.48,1047.0,17.52,4930.0,3904.0,79.19,1026.0,20.81
1,"Montgomery County, Tennessee",47.0,125.0,134768.0,5901.0,5552.0,5751.0,5335.0,5340.0,4878.0,2863.0,2709.0,8875.0,7677.0,12635.0,11606.0,10923.0,11036.0,7088.0,7312.0,4093.0,4695.0,2889.0,3287.0,1121.0,2123.0,296.0,783.0,38329.0,77152.0,19287.0,10499.0,98611.0,0.731709,25848.0,0.191796,709.0,0.005261,2455.0,0.018216,287.0,0.002130,2939.0,0.021808,3919.0,0.029080,6960.0,0.051644,39253.0,0.291264,48330.0,35964.0,28371.0,7593.0,1701.0,5892.0,12366.0,9740.0,2626.0,2.70,52167.0,48330.0,92.64,3837.0,7.36,48330.0,30700.0,63.52,17630.0,36.48
2,"Houston County, Tennessee",47.0,83.0,8088.0,268.0,273.0,311.0,243.0,266.0,298.0,174.0,137.0,307.0,285.0,520.0,484.0,555.0,548.0,546.0,567.0,467.0,489.0,343.0,384.0,189.0,264.0,53.0,117.0,1970.0,3812.0,2306.0,1350.0,7650.0,0.945846,268.0,0.033136,15.0,0.001855,10.0,0.001236,5.0,0.000618,63.0,0.007789,77.0,0.009520,101.0,0.012488,477.0,0.058976,3216.0,2300.0,1833.0,467.0,134.0,333.0,916.0,815.0,101.0,2.46,3901.0,3216.0,82.44,685.0,17.56,3216.0,2476.0,76.99,740.0,23.01
3,"Humphreys County, Tennessee",47.0,85.0,17929.0,567.0,505.0,610.0,597.0,660.0,584.0,402.0,357.0,712.0,654.0,1073.0,1173.0,1352.0,1337.0,1259.0,1317.0,1066.0,1049.0,705.0,823.0,339.0,526.0,74.0,188.0,4282.0,8877.0,4770.0,2655.0,17125.0,0.955156,527.0,0.029394,48.0,0.002677,46.0,0.002566,2.0,0.000112,29.0,0.001617,152.0,0.008478,148.0,0.008255,899.0,0.050142,7238.0,5145.0,4144.0,1001.0,266.0,735.0,2093.0,1808.0,285.0,2.44,8482.0,7238.0,85.33,1244.0,14.67,7238.0,5641.0,77.94,1597.0,22.06
4,"Dickson County, Tennessee",47.0,43.0,43156.0,1538.0,1437.0,1686.0,1584.0,1726.0,1601.0,957.0,957.0,1744.0,1749.0,3010.0,3108.0,3535.0,3581.0,2828.0,2944.0,2036.0,2066.0,1304.0,1520.0,656.0,1041.0,138.0,410.0,11486.0,22499.0,9171.0,5069.0,40243.0,0.932501,1978.0,0.045834,172.0,0.003986,116.0,0.002688,5.0,0.000116,204.0,0.004727,438.0,0.010149,484.0,0.011215,3136.0,0.072667,16473.0,12175.0,9604.0,2571.0,670.0,1901.0,4298.0,3678.0,620.0,2.59,17614.0,16473.0,93.52,1141.0,6.48,16473.0,12539.0,76.12,3934.0,23.88
